In [4]:
"""
P(up) / P(down) by open position and time of day.
================================================================================

For each fixed-clock range, compare its close against the range immediately
before it:

    UP      close(t) >  high(t-1)
    DOWN    close(t) <  low(t-1)
    INSIDE  close(t) lands within the prior range - no resolution

The conditioning variable is OPEN POSITION - where this range opened inside the
prior range:

    open_position = (open(t) - low(t-1)) / (high(t-1) - low(t-1))

    0.0 = opened at the prior low
    0.5 = opened at the prior midpoint
    1.0 = opened at the prior high
    <0 or >1 = opened outside the prior range entirely

Everything used as a feature is known at the moment the range opens. Nothing
from inside the range being predicted is used.

Outputs three tables, each with in-sample and out-of-sample columns:

    by_open_position          timeframe x open-position bucket
    by_slot                   timeframe x clock slot
    by_slot_and_open_position timeframe x clock slot x open-position bucket

Usage:  edit CONFIG, then  python rangeprob_stats.py
"""

import numpy as np
import pandas as pd

# ============================================================================
# CONFIG
# ============================================================================

CSV_PATH = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
SYMBOL = "NQ"
OUT_DIR = "."

START_DATE = "2013-01-01"
SPLIT_DATE = "2019-06-01"      # in-sample before, out-of-sample from here

TIMEFRAMES = [15, 30, 60, 120, 240]

SESSION_START_HOUR = 18        # anchors resample bins to the futures open (ET)
MIN_COVERAGE = 0.60            # drop ranges missing >40% of their 1-min bars

# Open-position buckets. Edges are in units of the prior range.
#
# Bins are LEFT-CLOSED, right-open: [a, b). This matters at the boundaries.
# Opening exactly AT the prior low (open_position == 0) is common - the prior
# low is often the prior close - so it must not be swept into a "below pLow"
# bucket. Here it falls in "0.0-0.1", and opening exactly at the prior high
# falls in "pHigh or above". Labels below say exactly what each bin contains.
OP_EDGES = [-np.inf, 0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0, np.inf]
OP_LABELS = ["below pLow", "0.0-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4",
             "0.4-0.5", "0.5-0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9",
             "0.9-1.0", "pHigh or above"]

MIN_RESOLVED = 30              # suppress cells with fewer resolved observations
WRITE_EXCEL = True

# Console printing. The full tables always go to the CSVs regardless.
PRINT_TABLES = True            # print slot x open-position grids per timeframe
PRINT_DOWN = True              # also print P(down|res); it is 100 - P(up|res)
PRINT_COUNTS = True            # print the resolved-sample-size grid alongside
PRINT_MIN_N = 30               # blank a cell in the printed grid below this n


# ============================================================================
# DATA
# ============================================================================

def load_1min(path=CSV_PATH, start=START_DATE):
    """Load the 1-minute CSV into a datetime-indexed OHLC frame."""
    df = pd.read_csv(path)
    df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"])
    df = df.set_index("Datetime").sort_index()
    return df[["Open", "High", "Low", "Close"]].loc[start:]


def build_ranges(df_1min, tf, session_start_hour=SESSION_START_HOUR,
                 min_coverage=MIN_COVERAGE):
    """
    Resample to fixed tf-minute ranges anchored to the session start hour.

    Two filters matter for correctness:
      - coverage: a range missing most of its 1-minute bars (holidays, gaps)
        has an unreliable high/low and is dropped
      - adjacency: a range is only usable if the previous SURVIVING range
        starts exactly one timeframe earlier, so weekend and holiday
        boundaries never become "prior range"
    """
    first = df_1min.index[0]
    origin = pd.Timestamp(first.year, first.month, first.day, session_start_hour)
    if origin > first:
        origin -= pd.Timedelta(days=1)

    rs = df_1min.resample(f"{tf}min", origin=origin)
    b = rs.agg({"Open": "first", "High": "max", "Low": "min", "Close": "last"})
    coverage = rs["Close"].count() / float(tf)

    b = b.dropna(subset=["Open", "High", "Low", "Close"])
    b = b[coverage.reindex(b.index) >= min_coverage].copy()

    idx = pd.Series(b.index, index=b.index)
    b["prev_adjacent"] = (idx - idx.shift(1)) == pd.Timedelta(minutes=tf)
    return b


def build_pairs(b, tf):
    """One row per range, with the prior range's geometry and the outcome."""
    d = pd.DataFrame({
        "open": b["Open"],
        "close": b["Close"],
        "prior_high": b["High"].shift(1),
        "prior_low": b["Low"].shift(1),
    })
    d["prior_range"] = d["prior_high"] - d["prior_low"]
    d = d[b["prev_adjacent"].fillna(False)].dropna()
    d = d[d["prior_range"] > 0]

    d["open_position"] = (d["open"] - d["prior_low"]) / d["prior_range"]
    d["up"] = d["close"] > d["prior_high"]
    d["down"] = d["close"] < d["prior_low"]
    d["resolved"] = d["up"] | d["down"]

    d["timeframe"] = f"{tf}M"
    d["slot_ET"] = d.index.strftime("%H:%M")
    d["op_bucket"] = pd.cut(d["open_position"], OP_EDGES, labels=OP_LABELS,
                            right=False)
    d["period"] = np.where(d.index < SPLIT_DATE, "IS", "OOS")
    return d


# ============================================================================
# STATS
# ============================================================================

def wilson(k, n, z=1.96):
    """
    Wilson score interval - stays sensible on small and lopsided cells.
    Accepts scalars or numpy arrays.
    """
    k = np.asarray(k, dtype=float)
    n = np.asarray(n, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(n > 0, k / n, np.nan)
        den = 1 + z * z / n
        centre = (p + z * z / (2 * n)) / den
        half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return centre - half, centre + half


def cell_stats(d, keys):
    """
    All outcome statistics for each group of ranges.

    Uses groupby.agg rather than groupby.apply so it works on every pandas
    version (apply's include_groups argument only exists in 2.2+) and so the
    whole thing stays vectorised.
    """
    g = d.groupby(keys, observed=True)
    out = pd.DataFrame({
        "ranges": g.size(),
        "up_n": g["up"].sum(),
        "down_n": g["down"].sum(),
        "resolved_n": g["resolved"].sum(),
    })

    n = out["ranges"].to_numpy(dtype=float)
    nr = out["resolved_n"].to_numpy(dtype=float)
    k = out["up_n"].to_numpy(dtype=float)

    with np.errstate(divide="ignore", invalid="ignore"):
        p_up = np.where(nr > 0, k / nr, np.nan)
        out["up_pct"] = 100 * out["up_n"].to_numpy(dtype=float) / n
        out["down_pct"] = 100 * out["down_n"].to_numpy(dtype=float) / n
        out["resolved_pct"] = 100 * nr / n
    out["inside_pct"] = 100 - out["resolved_pct"]
    out["p_up_given_resolved"] = 100 * p_up
    out["p_down_given_resolved"] = 100 * (1 - p_up)

    lo, hi = wilson(k, nr)
    out["ci_lo"] = 100 * lo
    out["ci_hi"] = 100 * hi

    return out[["ranges", "up_pct", "down_pct", "inside_pct", "resolved_pct",
                "resolved_n", "p_up_given_resolved", "p_down_given_resolved",
                "ci_lo", "ci_hi"]]


def summarise(d, keys, min_resolved=MIN_RESOLVED):
    """
    Group by `keys` and compute stats for the full sample, then attach the
    in-sample and out-of-sample versions of the headline probability so the
    stability of each cell is visible per row.
    """
    full = cell_stats(d, keys)

    halves = {}
    for period in ("IS", "OOS"):
        sub = d[d["period"] == period]
        s = cell_stats(sub, keys)
        halves[period] = s[["resolved_n", "p_up_given_resolved"]].rename(
            columns={"resolved_n": f"{period.lower()}_n",
                     "p_up_given_resolved": f"p_up_{period.lower()}"})

    out = full.join(halves["IS"]).join(halves["OOS"])
    out["oos_minus_is"] = out["p_up_oos"] - out["p_up_is"]
    out = out[out["resolved_n"] >= min_resolved]

    for c in ["ranges", "resolved_n", "is_n", "oos_n"]:
        if c in out:
            out[c] = out[c].fillna(0).astype(int)
    return out.reset_index()


# ============================================================================
# OUTPUT
# ============================================================================

def sort_tf(df):
    df = df.copy()
    df["_t"] = df["timeframe"].str.rstrip("M").astype(int)
    cols = ["_t"] + [c for c in ["slot_ET", "op_bucket"] if c in df.columns]
    return df.sort_values(cols).drop(columns="_t")


def write_excel(tables, path):
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    ws = wb.active
    ws.title = "Read me"
    lines = [
        (f"{SYMBOL} range direction by open position and time of day", True),
        ("", False),
        (f"Source: {CSV_PATH}", False),
        (f"Period: {START_DATE} onward. In-sample before {SPLIT_DATE}, "
         f"out-of-sample from {SPLIT_DATE}.", False),
        ("", False),
        ("OUTCOMES (this range vs the one immediately before it)", True),
        ("up_pct       close is ABOVE the prior range high", False),
        ("down_pct     close is BELOW the prior range low", False),
        ("inside_pct   close lands inside the prior range - no resolution", False),
        ("p_up_given_resolved    up_pct / (up_pct + down_pct)", False),
        ("p_down_given_resolved  100 - p_up_given_resolved", False),
        ("ci_lo / ci_hi          95% Wilson interval on p_up_given_resolved", False),
        ("", False),
        ("OPEN POSITION", True),
        ("(open - prior low) / (prior high - prior low)", False),
        ("0.0 = opened at the prior low, 1.0 = opened at the prior high", False),
        ("", False),
        ("STABILITY", True),
        ("p_up_is / p_up_oos are the same statistic on each half of the sample;", False),
        ("oos_minus_is is the drift. Open-position buckets hold to within a point", False),
        ("or two. Clock-slot effects are far less stable - check this column", False),
        ("before relying on any single slot.", False),
        ("", False),
        ("Descriptive historical statistics, not a trading recommendation.", False),
    ]
    for i, (t, bold) in enumerate(lines, 1):
        c = ws.cell(row=i, column=1, value=t)
        c.font = Font(name="Arial", size=11, bold=bold)
    ws.column_dimensions["A"].width = 92

    for name, df in tables.items():
        s = wb.create_sheet(name[:31])
        for j, col in enumerate(df.columns, 1):
            c = s.cell(row=1, column=j, value=col)
            c.font = Font(name="Arial", size=10, bold=True)
            c.fill = PatternFill("solid", fgColor="D9D9D9")
            c.alignment = Alignment(horizontal="center", wrap_text=True,
                                    vertical="center")
        for i, (_, row) in enumerate(df.iterrows(), 2):
            for j, col in enumerate(df.columns, 1):
                v = row[col]
                c = s.cell(row=i, column=j,
                           value=None if pd.isna(v) else
                           (float(v) if isinstance(v, (int, float, np.floating,
                                                       np.integer)) else str(v)))
                c.font = Font(name="Arial", size=10)
                if isinstance(v, (float, np.floating)):
                    c.number_format = "0.0"
                elif isinstance(v, (int, np.integer)):
                    c.number_format = "#,##0"
        s.freeze_panes = "A2"
        s.row_dimensions[1].height = 30
        for j, col in enumerate(df.columns, 1):
            s.column_dimensions[get_column_letter(j)].width = max(10, min(22, len(col) + 4))
    wb.save(path)


def print_slot_grids(t, timeframes=TIMEFRAMES):
    """
    For each timeframe, print a slot x open-position grid of P(up|resolved),
    optionally P(down|resolved) and the resolved sample size.

    Cells thinner than PRINT_MIN_N are blanked so a 3-observation cell reading
    100% never sits in the grid looking like a finding.
    """
    order = [l for l in OP_LABELS]

    def grid(df, value, fmt):
        p = df.pivot(index="slot_ET", columns="op_bucket", values=value)
        cols = [c for c in order if c in p.columns]
        p = p[cols].sort_index()
        n = df.pivot(index="slot_ET", columns="op_bucket",
                     values="resolved_n")[cols].sort_index()
        p = p.where(n >= PRINT_MIN_N)
        return p.round(fmt)

    for tf in timeframes:
        d = t[t["timeframe"] == f"{tf}M"]
        if d.empty:
            continue
        print("\n" + "=" * 100)
        print(f"{tf}M  -  P(up | resolved) %   rows = slot (ET), columns = open position")
        print("=" * 100)
        print(grid(d, "p_up_given_resolved", 1).to_string(na_rep="  -"))

        if PRINT_DOWN:
            print(f"\n{tf}M  -  P(down | resolved) %")
            print("-" * 100)
            print(grid(d, "p_down_given_resolved", 1).to_string(na_rep="  -"))

        if PRINT_COUNTS:
            print(f"\n{tf}M  -  resolved sample size per cell")
            print("-" * 100)
            print(grid(d, "resolved_n", 0).to_string(na_rep="  -"))


def main():
    print(f"Loading {CSV_PATH} ...")
    df_1min = load_1min()
    print(f"  {len(df_1min):,} 1-minute bars, "
          f"{df_1min.index.min()} -> {df_1min.index.max()}\n")

    frames = []
    for tf in TIMEFRAMES:
        b = build_ranges(df_1min, tf)
        d = build_pairs(b, tf)
        frames.append(d)
        print(f"  {tf:>4}M  {len(d):>8,} usable ranges   "
              f"resolved {100*d['resolved'].mean():.1f}%   "
              f"P(up|res) {100*d['up'][d['resolved']].mean():.1f}%")
    D = pd.concat(frames)

    tables = {
        "by_open_position": sort_tf(summarise(D, ["timeframe", "op_bucket"])),
        "by_slot": sort_tf(summarise(D, ["timeframe", "slot_ET"])),
        "by_slot_and_open_position": sort_tf(
            summarise(D, ["timeframe", "slot_ET", "op_bucket"])),
    }

    for name, t in tables.items():
        path = f"{OUT_DIR}/{SYMBOL.lower()}_{name}.csv"
        t.to_csv(path, index=False, float_format="%.3f")
        print(f"\n{name}: {len(t):,} cells -> {path}")

    if WRITE_EXCEL:
        xl = f"{OUT_DIR}/{SYMBOL.lower()}_range_direction.xlsx"
        write_excel(tables, xl)
        print(f"\nworkbook -> {xl}")

    print("\n" + "=" * 78)
    print("P(up | resolved) by open position, all timeframes")
    print("=" * 78)
    piv = tables["by_open_position"].pivot(
        index="op_bucket", columns="timeframe", values="p_up_given_resolved")
    piv = piv.reindex([l for l in OP_LABELS if l in piv.index])
    piv = piv[[f"{t}M" for t in TIMEFRAMES if f"{t}M" in piv.columns]]
    print(piv.round(1).to_string())

    if PRINT_TABLES:
        print_slot_grids(tables["by_slot_and_open_position"])

    return tables


if __name__ == "__main__":
    main()

Loading C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv ...
  4,161,727 1-minute bars, 2013-01-02 06:00:00 -> 2025-03-14 16:59:00

    15M   273,275 usable ranges   resolved 45.7%   P(up|res) 52.9%
    30M   132,944 usable ranges   resolved 46.1%   P(up|res) 53.9%
    60M    67,179 usable ranges   resolved 47.0%   P(up|res) 55.2%
   120M    30,659 usable ranges   resolved 49.8%   P(up|res) 56.2%
   240M    17,739 usable ranges   resolved 49.2%   P(up|res) 57.4%

by_open_position: 59 cells -> ./nq_by_open_position.csv

by_slot: 175 cells -> ./nq_by_slot.csv

by_slot_and_open_position: 1,825 cells -> ./nq_by_slot_and_open_position.csv

workbook -> ./nq_range_direction.xlsx

P(up | resolved) by open position, all timeframes
timeframe        15M   30M   60M  120M  240M
op_bucket                                   
below pLow      12.1  12.6  12.5   NaN   8.7
0.0-0.1         14.7  14.7  15.5  18.7  17.8
0.1-0.2         20.4  21.3  21.1  23.6  23.6
0.2-0.3         29.1  29.7  30.3  32.1  33

op_bucket  below pLow  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                                        
00:00               -    106.0     94.0    112.0    112.0    106.0    130.0    135.0    144.0    134.0     68.0            98.0
00:15               -    125.0     94.0    120.0     90.0     96.0    139.0    102.0    124.0    125.0     68.0            77.0
00:30               -    108.0    119.0    139.0    108.0     99.0    146.0     97.0    129.0    135.0     87.0            94.0
00:45               -    119.0     90.0    108.0    108.0     77.0    102.0    129.0    110.0    148.0     82.0            76.0
01:00               -    126.0    128.0    154.0    132.0    141.0    156.0    158.0    139.0    157.0     79.0           124.0
01:15               -    135.0     99.0    124.0     96.0     86.0    100.0    112.0    119.0    149.0  

op_bucket  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                            
00:00         68.0    103.0    124.0     80.0    102.0    129.0    137.0    124.0    153.0     94.0            42.0
01:00         91.0    130.0    156.0    141.0    139.0    152.0    146.0    156.0    185.0    133.0            40.0
02:00        125.0    143.0    153.0    150.0    139.0    151.0    168.0    177.0    208.0    149.0            50.0
03:00        155.0    186.0    179.0    168.0    168.0    198.0    167.0    176.0    231.0    161.0            39.0
04:00        136.0    154.0    102.0    118.0    118.0    123.0    132.0    130.0    166.0    138.0               -
05:00        110.0    129.0    117.0     97.0    115.0    118.0    107.0    120.0    169.0    137.0            31.0
06:00        101.0    136.0    104.0    118.0    119.0    116.0    145.0

In [5]:
"""
P(up) / P(down) by open position and time of day.
================================================================================

For each fixed-clock range, compare its close against the range immediately
before it:

    UP      close(t) >  high(t-1)
    DOWN    close(t) <  low(t-1)
    INSIDE  close(t) lands within the prior range - no resolution

The conditioning variable is OPEN POSITION - where this range opened inside the
prior range:

    open_position = (open(t) - low(t-1)) / (high(t-1) - low(t-1))

    0.0 = opened at the prior low
    0.5 = opened at the prior midpoint
    1.0 = opened at the prior high
    <0 or >1 = opened outside the prior range entirely

Everything used as a feature is known at the moment the range opens. Nothing
from inside the range being predicted is used.

Outputs three tables, each with in-sample and out-of-sample columns:

    by_open_position          timeframe x open-position bucket
    by_slot                   timeframe x clock slot
    by_slot_and_open_position timeframe x clock slot x open-position bucket

Usage:  edit CONFIG, then  python rangeprob_stats.py
"""

import numpy as np
import pandas as pd

# ============================================================================
# CONFIG
# ============================================================================

CSV_PATH = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
SYMBOL = "NQ"
OUT_DIR = "."

START_DATE = "2013-01-01"
SPLIT_DATE = "2019-06-01"      # in-sample before, out-of-sample from here

TIMEFRAMES = [15, 30, 60, 120, 240]

SESSION_START_HOUR = 18        # anchors resample bins to the futures open (ET)
MIN_COVERAGE = 0.60            # drop ranges missing >40% of their 1-min bars

# Open-position buckets. Edges are in units of the prior range.
#
# Bins are LEFT-CLOSED, right-open: [a, b). This matters at the boundaries.
# Opening exactly AT the prior low (open_position == 0) is common - the prior
# low is often the prior close - so it must not be swept into a "below pLow"
# bucket. Here it falls in "0.0-0.1", and opening exactly at the prior high
# falls in "pHigh or above". Labels below say exactly what each bin contains.
OP_EDGES = [-np.inf, 0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0, np.inf]
OP_LABELS = ["below pLow", "0.0-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4",
             "0.4-0.5", "0.5-0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9",
             "0.9-1.0", "pHigh or above"]

MIN_RESOLVED = 30              # suppress cells with fewer resolved observations
WRITE_EXCEL = True

# Console printing. The full tables always go to the CSVs regardless.
PRINT_TABLES = True            # print slot x open-position grids per timeframe
PRINT_DOWN = True              # also print P(down|res); it is 100 - P(up|res)
PRINT_COUNTS = True            # print the resolved-sample-size grid alongside
PRINT_MIN_N = 30               # blank a cell in the printed grid below this n

# --- Volatility feature -----------------------------------------------------
# How big a typical move is, relative to the prior range. This is what drives
# whether a range resolves at all: a range that usually moves a small fraction
# of the prior range will usually close inside it.
VOL_WINDOW = 250               # completed ranges in the rolling average
VOL_EDGES = [-np.inf, .25, .35, .45, .60, np.inf]
VOL_LABELS = ["very quiet", "quiet", "normal", "active", "wide"]

# --- Walk-forward -----------------------------------------------------------
RUN_WALKFORWARD = True
WF_MIN_TRAIN = 2000            # ranges required before the first prediction
WF_MIN_CELL = 50               # cell falls back to the base rate below this n


# ============================================================================
# DATA
# ============================================================================

def load_1min(path=CSV_PATH, start=START_DATE):
    """Load the 1-minute CSV into a datetime-indexed OHLC frame."""
    df = pd.read_csv(path)
    df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"])
    df = df.set_index("Datetime").sort_index()
    return df[["Open", "High", "Low", "Close"]].loc[start:]


def build_ranges(df_1min, tf, session_start_hour=SESSION_START_HOUR,
                 min_coverage=MIN_COVERAGE):
    """
    Resample to fixed tf-minute ranges anchored to the session start hour.

    Two filters matter for correctness:
      - coverage: a range missing most of its 1-minute bars (holidays, gaps)
        has an unreliable high/low and is dropped
      - adjacency: a range is only usable if the previous SURVIVING range
        starts exactly one timeframe earlier, so weekend and holiday
        boundaries never become "prior range"
    """
    first = df_1min.index[0]
    origin = pd.Timestamp(first.year, first.month, first.day, session_start_hour)
    if origin > first:
        origin -= pd.Timedelta(days=1)

    rs = df_1min.resample(f"{tf}min", origin=origin)
    b = rs.agg({"Open": "first", "High": "max", "Low": "min", "Close": "last"})
    coverage = rs["Close"].count() / float(tf)

    b = b.dropna(subset=["Open", "High", "Low", "Close"])
    b = b[coverage.reindex(b.index) >= min_coverage].copy()

    idx = pd.Series(b.index, index=b.index)
    b["prev_adjacent"] = (idx - idx.shift(1)) == pd.Timedelta(minutes=tf)
    return b


def build_pairs(b, tf):
    """One row per range, with the prior range's geometry and the outcome."""
    d = pd.DataFrame({
        "open": b["Open"],
        "close": b["Close"],
        "prior_high": b["High"].shift(1),
        "prior_low": b["Low"].shift(1),
    })
    d["prior_range"] = d["prior_high"] - d["prior_low"]
    d = d[b["prev_adjacent"].fillna(False)].dropna()
    d = d[d["prior_range"] > 0]

    d["open_position"] = (d["open"] - d["prior_low"]) / d["prior_range"]
    d["up"] = d["close"] > d["prior_high"]
    d["down"] = d["close"] < d["prior_low"]
    d["resolved"] = d["up"] | d["down"]

    d["timeframe"] = f"{tf}M"
    d["slot_ET"] = d.index.strftime("%H:%M")
    d["op_bucket"] = pd.cut(d["open_position"], OP_EDGES, labels=OP_LABELS,
                            right=False)
    d["period"] = np.where(d.index < SPLIT_DATE, "IS", "OOS")

    # Typical recent move, as a multiple of the prior range. Known at the open:
    # the rolling mean is over COMPLETED ranges and then shifted one further,
    # so nothing from the range being predicted leaks in.
    move = (d["close"] - d["open"]).abs()
    d["vol_ratio"] = (move.rolling(VOL_WINDOW, min_periods=50).mean().shift(1)
                      / d["prior_range"])
    d["vol_bucket"] = pd.cut(d["vol_ratio"], VOL_EDGES, labels=VOL_LABELS,
                             right=False)
    return d


# ============================================================================
# STATS
# ============================================================================

def wilson(k, n, z=1.96):
    """
    Wilson score interval - stays sensible on small and lopsided cells.
    Accepts scalars or numpy arrays.
    """
    k = np.asarray(k, dtype=float)
    n = np.asarray(n, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(n > 0, k / n, np.nan)
        den = 1 + z * z / n
        centre = (p + z * z / (2 * n)) / den
        half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return centre - half, centre + half


def cell_stats(d, keys):
    """
    All outcome statistics for each group of ranges.

    Uses groupby.agg rather than groupby.apply so it works on every pandas
    version (apply's include_groups argument only exists in 2.2+) and so the
    whole thing stays vectorised.
    """
    g = d.groupby(keys, observed=True)
    out = pd.DataFrame({
        "ranges": g.size(),
        "up_n": g["up"].sum(),
        "down_n": g["down"].sum(),
        "resolved_n": g["resolved"].sum(),
    })

    n = out["ranges"].to_numpy(dtype=float)
    nr = out["resolved_n"].to_numpy(dtype=float)
    k = out["up_n"].to_numpy(dtype=float)

    with np.errstate(divide="ignore", invalid="ignore"):
        p_up = np.where(nr > 0, k / nr, np.nan)
        out["up_pct"] = 100 * out["up_n"].to_numpy(dtype=float) / n
        out["down_pct"] = 100 * out["down_n"].to_numpy(dtype=float) / n
        out["resolved_pct"] = 100 * nr / n
    out["inside_pct"] = 100 - out["resolved_pct"]
    out["p_up_given_resolved"] = 100 * p_up
    out["p_down_given_resolved"] = 100 * (1 - p_up)

    lo, hi = wilson(k, nr)
    out["ci_lo"] = 100 * lo
    out["ci_hi"] = 100 * hi

    return out[["ranges", "up_pct", "down_pct", "inside_pct", "resolved_pct",
                "resolved_n", "p_up_given_resolved", "p_down_given_resolved",
                "ci_lo", "ci_hi"]]


def summarise(d, keys, min_resolved=MIN_RESOLVED):
    """
    Group by `keys` and compute stats for the full sample, then attach the
    in-sample and out-of-sample versions of the headline probability so the
    stability of each cell is visible per row.
    """
    full = cell_stats(d, keys)

    halves = {}
    for period in ("IS", "OOS"):
        sub = d[d["period"] == period]
        s = cell_stats(sub, keys)
        halves[period] = s[["resolved_n", "p_up_given_resolved"]].rename(
            columns={"resolved_n": f"{period.lower()}_n",
                     "p_up_given_resolved": f"p_up_{period.lower()}"})

    out = full.join(halves["IS"]).join(halves["OOS"])
    out["oos_minus_is"] = out["p_up_oos"] - out["p_up_is"]
    out = out[out["resolved_n"] >= min_resolved]

    for c in ["ranges", "resolved_n", "is_n", "oos_n"]:
        if c in out:
            out[c] = out[c].fillna(0).astype(int)
    return out.reset_index()


# ============================================================================
# OUTPUT
# ============================================================================

def sort_tf(df):
    df = df.copy()
    df["_t"] = df["timeframe"].str.rstrip("M").astype(int)
    cols = ["_t"] + [c for c in ["slot_ET", "op_bucket"] if c in df.columns]
    return df.sort_values(cols).drop(columns="_t")


def write_excel(tables, path):
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    ws = wb.active
    ws.title = "Read me"
    lines = [
        (f"{SYMBOL} range direction by open position and time of day", True),
        ("", False),
        (f"Source: {CSV_PATH}", False),
        (f"Period: {START_DATE} onward. In-sample before {SPLIT_DATE}, "
         f"out-of-sample from {SPLIT_DATE}.", False),
        ("", False),
        ("OUTCOMES (this range vs the one immediately before it)", True),
        ("up_pct       close is ABOVE the prior range high", False),
        ("down_pct     close is BELOW the prior range low", False),
        ("inside_pct   close lands inside the prior range - no resolution", False),
        ("p_up_given_resolved    up_pct / (up_pct + down_pct)", False),
        ("p_down_given_resolved  100 - p_up_given_resolved", False),
        ("ci_lo / ci_hi          95% Wilson interval on p_up_given_resolved", False),
        ("", False),
        ("OPEN POSITION", True),
        ("(open - prior low) / (prior high - prior low)", False),
        ("0.0 = opened at the prior low, 1.0 = opened at the prior high", False),
        ("", False),
        ("STABILITY", True),
        ("p_up_is / p_up_oos are the same statistic on each half of the sample;", False),
        ("oos_minus_is is the drift. Open-position buckets hold to within a point", False),
        ("or two. Clock-slot effects are far less stable - check this column", False),
        ("before relying on any single slot.", False),
        ("", False),
        ("Descriptive historical statistics, not a trading recommendation.", False),
    ]
    for i, (t, bold) in enumerate(lines, 1):
        c = ws.cell(row=i, column=1, value=t)
        c.font = Font(name="Arial", size=11, bold=bold)
    ws.column_dimensions["A"].width = 92

    for name, df in tables.items():
        s = wb.create_sheet(name[:31])
        for j, col in enumerate(df.columns, 1):
            c = s.cell(row=1, column=j, value=col)
            c.font = Font(name="Arial", size=10, bold=True)
            c.fill = PatternFill("solid", fgColor="D9D9D9")
            c.alignment = Alignment(horizontal="center", wrap_text=True,
                                    vertical="center")
        for i, (_, row) in enumerate(df.iterrows(), 2):
            for j, col in enumerate(df.columns, 1):
                v = row[col]
                c = s.cell(row=i, column=j,
                           value=None if pd.isna(v) else
                           (float(v) if isinstance(v, (int, float, np.floating,
                                                       np.integer)) else str(v)))
                c.font = Font(name="Arial", size=10)
                if isinstance(v, (float, np.floating)):
                    c.number_format = "0.0"
                elif isinstance(v, (int, np.integer)):
                    c.number_format = "#,##0"
        s.freeze_panes = "A2"
        s.row_dimensions[1].height = 30
        for j, col in enumerate(df.columns, 1):
            s.column_dimensions[get_column_letter(j)].width = max(10, min(22, len(col) + 4))
    wb.save(path)


def print_slot_grids(t, timeframes=TIMEFRAMES):
    """
    For each timeframe, print a slot x open-position grid of P(up|resolved),
    optionally P(down|resolved) and the resolved sample size.

    Cells thinner than PRINT_MIN_N are blanked so a 3-observation cell reading
    100% never sits in the grid looking like a finding.
    """
    order = [l for l in OP_LABELS]

    def grid(df, value, fmt):
        p = df.pivot(index="slot_ET", columns="op_bucket", values=value)
        cols = [c for c in order if c in p.columns]
        p = p[cols].sort_index()
        n = df.pivot(index="slot_ET", columns="op_bucket",
                     values="resolved_n")[cols].sort_index()
        p = p.where(n >= PRINT_MIN_N)
        return p.round(fmt)

    for tf in timeframes:
        d = t[t["timeframe"] == f"{tf}M"]
        if d.empty:
            continue
        print("\n" + "=" * 100)
        print(f"{tf}M  -  P(up | resolved) %   rows = slot (ET), columns = open position")
        print("=" * 100)
        print(grid(d, "p_up_given_resolved", 1).to_string(na_rep="  -"))

        if PRINT_DOWN:
            print(f"\n{tf}M  -  P(down | resolved) %")
            print("-" * 100)
            print(grid(d, "p_down_given_resolved", 1).to_string(na_rep="  -"))

        if PRINT_COUNTS:
            print(f"\n{tf}M  -  resolved sample size per cell")
            print("-" * 100)
            print(grid(d, "resolved_n", 0).to_string(na_rep="  -"))


def walk_forward(d, min_train=WF_MIN_TRAIN, min_cell=WF_MIN_CELL):
    """
    Expanding-window walk-forward, refit at the start of each calendar year.

    Two models, both estimated only from ranges that closed before the year
    being predicted:

      P(resolve)      will this range close outside the prior range at all?
                      estimated from open position x volatility bucket
      P(up | resolve) if it does, which side?
                      estimated from open position

    Combined at the range open into the three probabilities that actually
    describe what happens next and sum to 1:

      p_up     = P(resolve) * P(up | resolve)
      p_down   = P(resolve) * (1 - P(up | resolve))
      p_inside = 1 - P(resolve)
    """
    d = d.dropna(subset=["vol_ratio"]).copy()
    d["_year"] = d.index.year
    out = []

    for year in sorted(d["_year"].unique()):
        train = d[d["_year"] < year]
        if len(train) < min_train:
            continue
        test = d[d["_year"] == year].copy()

        # P(up | resolved), by open-position bucket
        tr_res = train[train["resolved"]]
        base_up = tr_res["up"].mean()
        t_up = tr_res.groupby("op_bucket", observed=True)["up"].agg(["mean", "size"])
        t_up.loc[t_up["size"] < min_cell, "mean"] = base_up

        # P(resolve), by open-position x volatility bucket
        base_res = train["resolved"].mean()
        t_res = train.groupby(["op_bucket", "vol_bucket"],
                              observed=True)["resolved"].agg(["mean", "size"])
        t_res.loc[t_res["size"] < min_cell, "mean"] = base_res

        test["p_up_given_resolved"] = (
            test["op_bucket"].map(t_up["mean"]).astype(float).fillna(base_up))

        key = pd.MultiIndex.from_frame(test[["op_bucket", "vol_bucket"]])
        test["p_resolve"] = pd.Series(
            t_res["mean"].reindex(key).to_numpy(), index=test.index).fillna(base_res)

        test["p_up"] = test["p_resolve"] * test["p_up_given_resolved"]
        test["p_down"] = test["p_resolve"] * (1 - test["p_up_given_resolved"])
        test["p_inside"] = 1 - test["p_resolve"]
        out.append(test)

    return pd.concat(out) if out else pd.DataFrame()


def score_walk_forward(p, label):
    """Print accuracy and calibration for one timeframe's walk-forward run."""
    res = p[p["resolved"]]
    dir_acc = ((res["p_up_given_resolved"] > .5) == res["up"]).mean()
    dir_base = max(res["up"].mean(), 1 - res["up"].mean())
    brier_dir = ((res["up"] - res["p_up_given_resolved"]) ** 2).mean()

    # three-way: pick the largest of the three probabilities
    P = p[["p_down", "p_inside", "p_up"]].to_numpy()
    pick = P.argmax(axis=1)
    truth = np.where(p["up"], 2, np.where(p["down"], 0, 1))
    three_acc = (pick == truth).mean()
    three_base = pd.Series(truth).value_counts(normalize=True).max()

    # multiclass Brier on the three outcomes
    onehot = np.zeros_like(P)
    onehot[np.arange(len(truth)), truth] = 1
    brier3 = ((P - onehot) ** 2).sum(axis=1).mean()

    res_brier = ((p["resolved"] - p["p_resolve"]) ** 2).mean()

    print(f"{label:<7}{len(p):>10,}{len(res):>10,}"
          f"{100*dir_acc:>10.1f}%{100*dir_base:>9.1f}%{brier_dir:>9.4f}"
          f"{100*three_acc:>10.1f}%{100*three_base:>9.1f}%{brier3:>9.4f}{res_brier:>10.4f}")


def calibration(p, col, truth_col, bins, label):
    """Predicted vs realised, in probability bins."""
    print(f"\n{label}")
    print(f"{'predicted':<14}{'n':>9}{'predicted %':>13}{'actual %':>11}{'gap':>8}")
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (p[col] >= lo) & (p[col] < hi)
        if m.sum() < 50:
            continue
        pm, am = 100 * p[col][m].mean(), 100 * p[truth_col][m].mean()
        print(f"{f'{lo:.2f}-{hi:.2f}':<14}{int(m.sum()):>9,}{pm:>12.1f}%{am:>10.1f}%{am-pm:>+8.1f}")


def run_walk_forward(D):
    """Walk-forward every timeframe, print scores, return the predictions."""
    print("\n" + "=" * 104)
    print("WALK-FORWARD  -  expanding window, refit each January, nothing from the")
    print("range being predicted or from any later range is used.")
    print("=" * 104)
    print(f"{'TF':<7}{'ranges':>10}{'resolved':>10}{'dir acc':>11}{'base':>9}"
          f"{'Brier':>9}{'3-way acc':>10}{'base':>9}{'Brier3':>9}{'resBrier':>10}")

    preds = {}
    for tf in TIMEFRAMES:
        d = D[D["timeframe"] == f"{tf}M"]
        p = walk_forward(d)
        if p.empty:
            continue
        preds[tf] = p
        score_walk_forward(p, f"{tf}M")

    if preds:
        tf0 = sorted(preds)[len(preds) // 2]
        p = preds[tf0]
        calibration(p[p["resolved"]], "p_up_given_resolved", "up",
                    [0, .2, .35, .5, .65, .8, 1.01],
                    f"CALIBRATION of P(up | resolved), {tf0}M, resolved ranges only")
        calibration(p, "p_resolve", "resolved", [0, .35, .42, .48, .55, .65, 1.01],
                    f"CALIBRATION of P(resolve), {tf0}M, all ranges")
        calibration(p, "p_up", "up", [0, .12, .2, .28, .36, .45, 1.01],
                    f"CALIBRATION of P(up) unconditional, {tf0}M, all ranges")

        print("\nACCURACY BY YEAR, direction on resolved ranges")
        yrs = range(2015, 2026)
        print(f"{'TF':<7}" + "".join(f"{y:>8}" for y in yrs))
        for tf, p in sorted(preds.items()):
            r = p[p["resolved"]]
            g = ((r["p_up_given_resolved"] > .5) == r["up"]).groupby(r.index.year).mean()
            print(f"{str(tf)+'M':<7}" + "".join(
                f"{100*g[y]:>7.1f}%" if y in g.index else f"{'-':>8}" for y in yrs))

        print("\nWHAT YOU GET IF YOU COMMIT AT EVERY RANGE OPEN")
        print("(the unresolved ranges stop being free)")
        print(f"{'TF':<7}{'scored on resolved only':>26}{'counting unresolved as misses':>32}")
        for tf, p in sorted(preds.items()):
            r = p[p["resolved"]]
            a1 = ((r["p_up_given_resolved"] > .5) == r["up"]).mean()
            hit = (((p["p_up_given_resolved"] > .5) & p["up"]) |
                   ((p["p_up_given_resolved"] <= .5) & p["down"]))
            print(f"{str(tf)+'M':<7}{100*a1:>25.1f}%{100*hit.mean():>31.1f}%")

    return preds


def main():
    print(f"Loading {CSV_PATH} ...")
    df_1min = load_1min()
    print(f"  {len(df_1min):,} 1-minute bars, "
          f"{df_1min.index.min()} -> {df_1min.index.max()}\n")

    frames = []
    for tf in TIMEFRAMES:
        b = build_ranges(df_1min, tf)
        d = build_pairs(b, tf)
        frames.append(d)
        print(f"  {tf:>4}M  {len(d):>8,} usable ranges   "
              f"resolved {100*d['resolved'].mean():.1f}%   "
              f"P(up|res) {100*d['up'][d['resolved']].mean():.1f}%")
    D = pd.concat(frames)

    tables = {
        "by_open_position": sort_tf(summarise(D, ["timeframe", "op_bucket"])),
        "by_slot": sort_tf(summarise(D, ["timeframe", "slot_ET"])),
        "by_slot_and_open_position": sort_tf(
            summarise(D, ["timeframe", "slot_ET", "op_bucket"])),
    }

    for name, t in tables.items():
        path = f"{OUT_DIR}/{SYMBOL.lower()}_{name}.csv"
        t.to_csv(path, index=False, float_format="%.3f")
        print(f"\n{name}: {len(t):,} cells -> {path}")

    if WRITE_EXCEL:
        xl = f"{OUT_DIR}/{SYMBOL.lower()}_range_direction.xlsx"
        write_excel(tables, xl)
        print(f"\nworkbook -> {xl}")

    print("\n" + "=" * 78)
    print("P(up | resolved) by open position, all timeframes")
    print("=" * 78)
    piv = tables["by_open_position"].pivot(
        index="op_bucket", columns="timeframe", values="p_up_given_resolved")
    piv = piv.reindex([l for l in OP_LABELS if l in piv.index])
    piv = piv[[f"{t}M" for t in TIMEFRAMES if f"{t}M" in piv.columns]]
    print(piv.round(1).to_string())

    if PRINT_TABLES:
        print_slot_grids(tables["by_slot_and_open_position"])

    preds = {}
    if RUN_WALKFORWARD:
        preds = run_walk_forward(D)
        if preds:
            cols = ["timeframe", "slot_ET", "open_position", "op_bucket",
                    "vol_ratio", "vol_bucket", "p_resolve",
                    "p_up_given_resolved", "p_up", "p_down", "p_inside",
                    "up", "down", "resolved"]
            wf = pd.concat(preds.values())[cols].sort_index()
            path = f"{OUT_DIR}/{SYMBOL.lower()}_walkforward_predictions.csv"
            wf.to_csv(path, float_format="%.4f")
            print(f"\nwalk-forward predictions ({len(wf):,} rows) -> {path}")

    return tables, preds


if __name__ == "__main__":
    main()

Loading C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv ...
  4,161,727 1-minute bars, 2013-01-02 06:00:00 -> 2025-03-14 16:59:00

    15M   273,275 usable ranges   resolved 45.7%   P(up|res) 52.9%
    30M   132,944 usable ranges   resolved 46.1%   P(up|res) 53.9%
    60M    67,179 usable ranges   resolved 47.0%   P(up|res) 55.2%
   120M    30,659 usable ranges   resolved 49.8%   P(up|res) 56.2%
   240M    17,739 usable ranges   resolved 49.2%   P(up|res) 57.4%

by_open_position: 59 cells -> ./nq_by_open_position.csv

by_slot: 175 cells -> ./nq_by_slot.csv

by_slot_and_open_position: 1,825 cells -> ./nq_by_slot_and_open_position.csv

workbook -> ./nq_range_direction.xlsx

P(up | resolved) by open position, all timeframes
timeframe        15M   30M   60M  120M  240M
op_bucket                                   
below pLow      12.1  12.6  12.5   NaN   8.7
0.0-0.1         14.7  14.7  15.5  18.7  17.8
0.1-0.2         20.4  21.3  21.1  23.6  23.6
0.2-0.3         29.1  29.7  30.3  32.1  33

op_bucket  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                            
00:00         83.0     78.8     70.4     66.7     59.8     45.7     39.3     29.6     20.2     19.2            16.9
00:30         83.2     82.8     72.8     66.7     52.4     41.6     34.5     28.4     17.2     13.8            18.2
01:00         78.0     75.2     70.3     53.2     55.9     55.8     35.7     32.7     23.8     21.4            18.1
01:30         87.7     80.5     68.4     62.1     43.4     36.6     36.1     25.0     13.7     13.5            14.3
02:00         75.4     74.8     69.1     59.7     48.8     42.5     37.2     34.8     20.1     18.5            14.4
02:30         87.7     82.6     80.8     67.0     51.1     44.3     26.9     26.5     11.4     11.8            15.4
03:00         69.7     66.5     56.2     52.3     52.1     47.4     46.0

op_bucket  below pLow  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0
slot_ET                                                                                                        
02:00               -     62.9     62.4     59.6     58.0     54.8     45.6     37.4     31.6     27.3     25.9
06:00               -     80.8     64.3     56.7     51.2     52.0     45.9     31.2     35.6     31.0     26.8
10:00               -     82.5     78.3     67.3     51.9     48.3     47.0     33.7     32.3     19.9     14.2
14:00               -     91.8     90.1     83.3     63.4     47.7     49.3     25.8     17.0      9.7      4.0
18:00            97.3     95.3     97.3     91.5     68.4     52.8     52.6     23.1     11.8      5.7      2.3
22:00               -     90.8     91.9     77.6     67.5     51.6     34.8     29.7     15.0      9.0      5.0

240M  -  resolved sample size per cell
----------------------------------------------------------------

In [8]:
"""
Comprehensive range statistics.
================================================================================

For every fixed-clock range, measured against the range immediately before it,
and split by time of day and by where the range opened inside the prior range.

WHAT IS MEASURED
----------------

1. CLOSE OUTCOMES  (the rangeProb question)
     up        close > prior high
     down      close < prior low
     inside    close lands within the prior range

2. FIRST TOUCH  (which prior level price reaches first)
     hi_first  price trades above the prior high before trading below prior low
     lo_first  the reverse
     no_touch  neither level is reached
   Resolved at 1-MINUTE resolution, so same-bar ties are almost eliminated.

3. PATH STRUCTURE
     own_high_first   the range's own high forms before its own low
     minutes to first touch of each prior level

4. RETRACEMENTS AFTER A SWEEP
   Having touched the prior high first, does price come back within the same
   range to: the current open, the prior midpoint, the prior open, the prior
   low? And mirrored for a prior-low sweep.
   Also: does the sweep HOLD (close beyond the swept level) or REVERSE
   (close back beyond the prior midpoint)?

OUTPUT GRAIN
------------
   by_open_position            timeframe x open-position bucket
   by_slot                     timeframe x clock slot
   by_slot_and_open_position   timeframe x clock slot x open-position bucket

Every percentage carries a sample size. Cells below MIN_N are suppressed.

Usage: edit CONFIG, then  python range_stats_full.py
"""

import numpy as np
import pandas as pd

# ============================================================================
# CONFIG
# ============================================================================

CSV_PATH = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
SYMBOL = "NQ"
OUT_DIR = "."

START_DATE = "2013-01-01"
SPLIT_DATE = "2019-06-01"          # stability check: before vs from this date

TIMEFRAMES = [15, 30, 60, 120, 240]
SESSION_START_HOUR = 18
MIN_COVERAGE = 0.60

OP_EDGES = [-np.inf, 0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0, np.inf]
OP_LABELS = ["below pLow", "0.0-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4",
             "0.4-0.5", "0.5-0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9",
             "0.9-1.0", "pHigh or above"]

MIN_N = 50                          # suppress cells thinner than this
MIN_COND_N = 50                     # ...and conditional metrics thinner than this
                                    # in their OWN denominator (e.g. hs_* are only
                                    # defined on ranges that swept the prior high)
WRITE_EXCEL = True

# --- Console printing (the full tables always go to the CSVs regardless) -----
PRINT_SUMMARY = True                # open-position pivots
PRINT_BY_SLOT = True                # one row per clock slot, all key metrics
PRINT_SLOT_GRIDS = True             # slot x open-position grid, per metric below
PRINT_GRID_MIN_N = 50               # blank a grid cell below this sample size

# Metrics printed as slot x open-position grids. Add or remove freely - every
# entry adds len(TIMEFRAMES) grids to the output. Default covers all 21.
GRID_METRICS = [
    ("up_pct",                 "closes ABOVE prior high %"),
    ("down_pct",               "closes BELOW prior low %"),
    ("inside_pct",             "closes INSIDE prior range %"),
    ("p_up_given_resolved",    "P(closes above prior high | closed outside) %"),
    ("hi_first_pct",           "touched prior HIGH first %"),
    ("lo_first_pct",           "touched prior LOW first %"),
    ("no_touch_pct",           "neither prior level touched %"),
    ("p_hi_first_given_touch", "P(prior HIGH touched first | one was touched) %"),
    ("own_high_first_pct",     "range's own HIGH formed before its own LOW %"),
    ("min_to_hi_touch",        "median minutes to touching the prior high"),
    ("min_to_lo_touch",        "median minutes to touching the prior low"),
    ("hs_ret_curr_open",       "swept prior high, retraced to THIS range's open %"),
    ("hs_ret_prior_mid",       "swept prior high, retraced to prior MID %"),
    ("hs_ret_prior_open",      "swept prior high, retraced to prior OPEN %"),
    ("hs_ret_prior_low",       "swept prior high, retraced to prior LOW %"),
    ("hs_held",                "swept prior high, closed beyond it %"),
    ("hs_reversed",            "swept prior high, closed back below prior mid %"),
    ("ls_ret_curr_open",       "swept prior low, retraced to THIS range's open %"),
    ("ls_ret_prior_mid",       "swept prior low, retraced to prior MID %"),
    ("ls_ret_prior_open",      "swept prior low, retraced to prior OPEN %"),
    ("ls_ret_prior_high",      "swept prior low, retraced to prior HIGH %"),
    ("ls_held",                "swept prior low, closed beyond it %"),
    ("ls_reversed",            "swept prior low, closed back above prior mid %"),
]
PRINT_GRID_COUNTS = True            # print the denominator grid under each metric

# Columns shown in the per-slot table.
SLOT_COLS = ["ranges", "up_pct", "down_pct", "inside_pct", "p_up_given_resolved",
             "hi_first_pct", "lo_first_pct", "no_touch_pct",
             "p_hi_first_given_touch", "own_high_first_pct",
             "min_to_hi_touch", "min_to_lo_touch",
             "hs_ret_prior_mid", "hs_held", "hs_reversed",
             "ls_ret_prior_mid", "ls_held", "ls_reversed"]


# ============================================================================
# RANGE CONSTRUCTION
# ============================================================================

def load_1min(path=CSV_PATH, start=START_DATE):
    df = pd.read_csv(path)
    df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"])
    df = df.set_index("Datetime").sort_index()
    return df[["Open", "High", "Low", "Close"]].loc[start:]


def build_ranges(df_1min, tf):
    """Fixed tf-minute ranges anchored to the session start, coverage-filtered."""
    first = df_1min.index[0]
    origin = pd.Timestamp(first.year, first.month, first.day, SESSION_START_HOUR)
    if origin > first:
        origin -= pd.Timedelta(days=1)
    rs = df_1min.resample(f"{tf}min", origin=origin)
    b = rs.agg({"Open": "first", "High": "max", "Low": "min", "Close": "last"})
    cov = rs["Close"].count() / float(tf)
    b = b.dropna(subset=["Open", "High", "Low", "Close"])
    b = b[cov.reindex(b.index) >= MIN_COVERAGE].copy()
    idx = pd.Series(b.index, index=b.index)
    b["prev_adjacent"] = (idx - idx.shift(1)) == pd.Timedelta(minutes=tf)
    return b


def build_metrics(df_1min, tf):
    """
    One row per range, carrying prior-range geometry, close outcome, first
    touch, path structure and retracement flags.

    The intrabar work is done by mapping every 1-minute bar to the range that
    contains it, then reducing with groupby - no Python loop over ranges.
    """
    src = build_ranges(df_1min, tf)
    b = src[src["prev_adjacent"].fillna(False)].copy()

    for name, col in [("pH", "High"), ("pL", "Low"), ("pO", "Open"), ("pC", "Close")]:
        b[name] = src[col].shift(1).reindex(b.index)
    b["pR"] = b["pH"] - b["pL"]
    b = b.dropna(subset=["pH", "pL", "pO", "pR"])
    b = b[b["pR"] > 0]
    b["pM"] = (b["pH"] + b["pL"]) / 2
    b["open_position"] = (b["Open"] - b["pL"]) / b["pR"]

    b = b.reset_index().rename(columns={"index": "start", "Datetime": "start"})
    starts = b["start"].values
    n = len(b)

    # --- map 1-minute bars to ranges ---
    pos = np.searchsorted(starts, df_1min.index.values, side="right") - 1
    ok = pos >= 0
    within = np.zeros(len(df_1min), bool)
    within[ok] = (df_1min.index.values[ok] - starts[pos[ok]]) < np.timedelta64(tf, "m")
    sel = ok & within
    mm = pd.DataFrame({"rid": pos[sel],
                       "hi": df_1min["High"].values[sel],
                       "lo": df_1min["Low"].values[sel]})
    mm["seq"] = mm.groupby("rid").cumcount()
    g = mm.groupby("rid")

    pH = b["pH"].values[mm["rid"]]
    pL = b["pL"].values[mm["rid"]]

    # --- first touch of each prior level, in minutes from the range open ---
    b["t_up"] = mm["seq"].where(mm["hi"].values > pH).groupby(mm["rid"]).min() \
                         .reindex(range(n)).values
    b["t_dn"] = mm["seq"].where(mm["lo"].values < pL).groupby(mm["rid"]).min() \
                         .reindex(range(n)).values

    # --- when the range's own extremes formed ---
    b["t_own_high"] = g["hi"].idxmax().map(mm["seq"]).reindex(range(n)).values
    b["t_own_low"] = g["lo"].idxmin().map(mm["seq"]).reindex(range(n)).values

    # --- how far price pulled back AFTER the first touch, same range ---
    tu = b["t_up"].values[mm["rid"]]
    td = b["t_dn"].values[mm["rid"]]
    lo_after_up = pd.Series(np.where(mm["seq"].values >= tu, mm["lo"].values, np.nan)) \
                    .groupby(mm["rid"].values).min()
    hi_after_dn = pd.Series(np.where(mm["seq"].values >= td, mm["hi"].values, np.nan)) \
                    .groupby(mm["rid"].values).max()
    b["lo_after_up"] = lo_after_up.reindex(range(n)).values
    b["hi_after_dn"] = hi_after_dn.reindex(range(n)).values

    # ---------------- flags ----------------
    b["up"] = b["Close"] > b["pH"]
    b["down"] = b["Close"] < b["pL"]
    b["resolved"] = b["up"] | b["down"]

    tup, tdn = b["t_up"], b["t_dn"]
    b["hi_first"] = (tup < tdn) | (tup.notna() & tdn.isna())
    b["lo_first"] = (tdn < tup) | (tdn.notna() & tup.isna())
    b["touched"] = b["hi_first"] | b["lo_first"]
    b["own_high_first"] = b["t_own_high"] < b["t_own_low"]

    # retracements after sweeping the prior high first
    h = b["hi_first"]
    for name, level in [("curr_open", b["Open"]), ("prior_mid", b["pM"]),
                        ("prior_open", b["pO"]), ("prior_low", b["pL"])]:
        b[f"hs_ret_{name}"] = np.where(h, b["lo_after_up"] <= level, np.nan)
    b["hs_held"] = np.where(h, b["Close"] > b["pH"], np.nan)      # sweep held
    b["hs_reversed"] = np.where(h, b["Close"] < b["pM"], np.nan)  # swept then failed

    # ...and after sweeping the prior low first
    l = b["lo_first"]
    for name, level in [("curr_open", b["Open"]), ("prior_mid", b["pM"]),
                        ("prior_open", b["pO"]), ("prior_high", b["pH"])]:
        b[f"ls_ret_{name}"] = np.where(l, b["hi_after_dn"] >= level, np.nan)
    b["ls_held"] = np.where(l, b["Close"] < b["pL"], np.nan)
    b["ls_reversed"] = np.where(l, b["Close"] > b["pM"], np.nan)

    b["timeframe"] = f"{tf}M"
    b["slot_ET"] = pd.DatetimeIndex(b["start"]).strftime("%H:%M")
    b["op_bucket"] = pd.cut(b["open_position"], OP_EDGES, labels=OP_LABELS, right=False)
    b["period"] = np.where(pd.DatetimeIndex(b["start"]) < SPLIT_DATE, "IS", "OOS")
    return b


# ============================================================================
# AGGREGATION
# ============================================================================

# metric -> (column, subset column or None). Subset restricts the denominator.
METRICS = [
    ("up_pct",                  "up",              None),
    ("down_pct",                "down",            None),
    ("inside_pct",              "resolved",        "INVERT"),
    ("p_up_given_resolved",     "up",              "resolved"),
    ("hi_first_pct",            "hi_first",        None),
    ("lo_first_pct",            "lo_first",        None),
    ("no_touch_pct",            "touched",         "INVERT"),
    ("p_hi_first_given_touch",  "hi_first",        "touched"),
    ("own_high_first_pct",      "own_high_first",  None),
    ("hs_ret_curr_open",        "hs_ret_curr_open",  "hi_first"),
    ("hs_ret_prior_mid",        "hs_ret_prior_mid",  "hi_first"),
    ("hs_ret_prior_open",       "hs_ret_prior_open", "hi_first"),
    ("hs_ret_prior_low",        "hs_ret_prior_low",  "hi_first"),
    ("hs_held",                 "hs_held",           "hi_first"),
    ("hs_reversed",             "hs_reversed",       "hi_first"),
    ("ls_ret_curr_open",        "ls_ret_curr_open",  "lo_first"),
    ("ls_ret_prior_mid",        "ls_ret_prior_mid",  "lo_first"),
    ("ls_ret_prior_open",       "ls_ret_prior_open", "lo_first"),
    ("ls_ret_prior_high",       "ls_ret_prior_high", "lo_first"),
    ("ls_held",                 "ls_held",           "lo_first"),
    ("ls_reversed",             "ls_reversed",       "lo_first"),
]


def aggregate(d, keys):
    """Wide table of every metric, one row per group."""
    grp = d.groupby(keys, observed=True)
    out = pd.DataFrame({"ranges": grp.size()})

    for name, col, subset in METRICS:
        if subset is None:
            out[name] = 100 * grp[col].mean()
        elif subset == "INVERT":
            out[name] = 100 * (1 - grp[col].mean())
        else:
            sub = d[d[subset].fillna(False).astype(bool)]
            if sub.empty:
                out[name] = np.nan
                continue
            sg = sub.groupby(keys, observed=True)[col]
            out[name] = 100 * sg.mean().reindex(out.index)
            if name in ("p_up_given_resolved", "p_hi_first_given_touch"):
                out[name.replace("p_", "n_").replace("_given", "")] = \
                    sg.size().reindex(out.index)

    # sample sizes behind the conditional blocks
    for tag, col in [("n_hi_first", "hi_first"), ("n_lo_first", "lo_first"),
                     ("n_resolved", "resolved"), ("n_touched", "touched")]:
        out[tag] = grp[col].sum()

    # A conditional metric is only meaningful on its own denominator. Blank the
    # cell when that denominator is thin, otherwise a 3-observation cell shows
    # up as a clean-looking 66.7%.
    denom = {"resolved": "n_resolved", "touched": "n_touched",
             "hi_first": "n_hi_first", "lo_first": "n_lo_first"}
    for name, _col, subset in METRICS:
        if subset in denom:
            out.loc[out[denom[subset]] < MIN_COND_N, name] = np.nan

    # median minutes to first touch
    for tag, col in [("min_to_hi_touch", "t_up"), ("min_to_lo_touch", "t_dn")]:
        out[tag] = grp[col].median()

    out = out[out["ranges"] >= MIN_N]
    for c in out.columns:
        if c.startswith("n_") or c == "ranges":
            out[c] = out[c].fillna(0).astype(int)
    return out.reset_index()


def add_stability(d, keys, table):
    """Attach the in-sample / out-of-sample split of the two headline rates."""
    for period in ("IS", "OOS"):
        sub = d[d["period"] == period]
        for name, col, cond in [("p_up_given_resolved", "up", "resolved"),
                                ("p_hi_first_given_touch", "hi_first", "touched")]:
            s = sub[sub[cond].fillna(False).astype(bool)]
            v = 100 * s.groupby(keys, observed=True)[col].mean()
            table = table.merge(
                v.rename(f"{name}_{period.lower()}").reset_index(), on=keys, how="left")
    for name in ("p_up_given_resolved", "p_hi_first_given_touch"):
        table[f"{name}_drift"] = table[f"{name}_oos"] - table[f"{name}_is"]
    return table


def sort_tf(df):
    df = df.copy()
    df["_t"] = df["timeframe"].str.rstrip("M").astype(int)
    cols = ["_t"] + [c for c in ["slot_ET", "op_bucket"] if c in df.columns]
    return df.sort_values(cols).drop(columns="_t")


# ============================================================================
# OUTPUT
# ============================================================================

README = [
    ("{sym} comprehensive range statistics", True),
    ("", False),
    ("Each range is compared against the range immediately before it.", False),
    ("Open position = (open - prior low) / (prior high - prior low).", False),
    ("0.0 opened at the prior low, 1.0 opened at the prior high.", False),
    ("", False),
    ("CLOSE OUTCOMES", True),
    ("up_pct / down_pct / inside_pct    close above / below / inside the prior range", False),
    ("p_up_given_resolved               of the ranges closing outside, share closing above", False),
    ("", False),
    ("FIRST TOUCH  (1-minute resolution)", True),
    ("hi_first_pct    price traded above the prior high before below the prior low", False),
    ("lo_first_pct    the reverse", False),
    ("no_touch_pct    neither prior level was reached", False),
    ("p_hi_first_given_touch   of the ranges touching something, share touching the high first", False),
    ("min_to_hi_touch / min_to_lo_touch   median minutes from the range open to that touch", False),
    ("", False),
    ("PATH STRUCTURE", True),
    ("own_high_first_pct   the range's own high formed before its own low", False),
    ("", False),
    ("RETRACEMENT AFTER A SWEEP", True),
    ("hs_* applies to ranges that touched the prior HIGH first (n_hi_first).", False),
    ("ls_* applies to ranges that touched the prior LOW first (n_lo_first).", False),
    ("hs_ret_curr_open    after the sweep, price returned to this range's open", False),
    ("hs_ret_prior_mid    ...to the prior range midpoint", False),
    ("hs_ret_prior_open   ...to the prior range open", False),
    ("hs_ret_prior_low    ...all the way to the opposite prior extreme", False),
    ("hs_held             the range closed beyond the swept level", False),
    ("hs_reversed         the range closed back beyond the prior midpoint", False),
    ("All retracement measures are within the SAME range, after the touch.", False),
    ("", False),
    ("STABILITY", True),
    ("_is / _oos are the same rate on each half of the sample; _drift is the change.", False),
    ("Open-position effects hold to within a point or two. Clock-slot effects drift", False),
    ("much more - check before relying on a single slot.", False),
    ("", False),
    ("Descriptive historical statistics, not a trading recommendation.", False),
]


def write_excel(tables, path, symbol):
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    ws = wb.active
    ws.title = "Read me"
    for i, (t, bold) in enumerate(README, 1):
        c = ws.cell(row=i, column=1, value=t.format(sym=symbol))
        c.font = Font(name="Arial", size=11, bold=bold)
    ws.column_dimensions["A"].width = 96

    for name, df in tables.items():
        s = wb.create_sheet(name[:31])
        for j, col in enumerate(df.columns, 1):
            c = s.cell(row=1, column=j, value=col)
            c.font = Font(name="Arial", size=9, bold=True)
            c.fill = PatternFill("solid", fgColor="D9D9D9")
            c.alignment = Alignment(horizontal="center", wrap_text=True, vertical="center")
        for i, (_, row) in enumerate(df.iterrows(), 2):
            for j, col in enumerate(df.columns, 1):
                v = row[col]
                if pd.isna(v):
                    continue
                c = s.cell(row=i, column=j,
                           value=float(v) if isinstance(v, (float, np.floating))
                           else (int(v) if isinstance(v, (int, np.integer)) else str(v)))
                c.font = Font(name="Arial", size=9)
                c.number_format = "0.0" if isinstance(v, (float, np.floating)) else "#,##0"
        s.freeze_panes = "B2" if "slot_ET" not in df.columns else "C2"
        s.row_dimensions[1].height = 34
        for j, col in enumerate(df.columns, 1):
            s.column_dimensions[get_column_letter(j)].width = max(9, min(16, len(col) + 2))
    wb.save(path)


def print_by_slot(t, timeframes=TIMEFRAMES):
    """One row per clock slot, all key metrics, per timeframe."""
    cols = [c for c in SLOT_COLS if c in t.columns]
    for tf in timeframes:
        d = t[t["timeframe"] == f"{tf}M"]
        if d.empty:
            continue
        print("\n" + "=" * 100)
        print(f"{tf}M  -  BY TIME OF DAY (all open positions pooled)")
        print("=" * 100)
        print(d.set_index("slot_ET")[cols].sort_index().round(1).to_string(na_rep="  -"))


def print_slot_grids(t, timeframes=TIMEFRAMES):
    """slot x open-position grid, one per metric per timeframe."""
    order = [l for l in OP_LABELS]
    # each metric's true denominator column
    dcol = {}
    denom = {"resolved": "n_resolved", "touched": "n_touched",
             "hi_first": "n_hi_first", "lo_first": "n_lo_first"}
    for name, _c, subset in METRICS:
        dcol[name] = denom.get(subset, "ranges")

    for metric, title in GRID_METRICS:
        if metric not in t.columns:
            continue
        nc = dcol.get(metric, "ranges")
        for tf in timeframes:
            d = t[t["timeframe"] == f"{tf}M"]
            if d.empty:
                continue
            p = d.pivot(index="slot_ET", columns="op_bucket", values=metric)
            n = d.pivot(index="slot_ET", columns="op_bucket", values=nc)
            cols = [c for c in order if c in p.columns]
            p, n = p[cols].sort_index(), n[cols].sort_index()
            p = p.where(n >= PRINT_GRID_MIN_N)
            if p.notna().sum().sum() == 0:
                continue
            print("\n" + "=" * 100)
            print(f"{tf}M  -  {title}")
            print(f"rows = slot (ET), columns = open position   "
                  f"[cells with fewer than {PRINT_GRID_MIN_N} in '{nc}' are blank]")
            print("=" * 100)
            print(p.round(1).to_string(na_rep="  -"))
            if PRINT_GRID_COUNTS:
                print(f"\n  sample size ({nc}):")
                print(n.where(n >= PRINT_GRID_MIN_N).round(0).to_string(na_rep="  -"))


def main():
    print(f"Loading {CSV_PATH} ...")
    df_1min = load_1min()
    print(f"  {len(df_1min):,} 1-minute bars, "
          f"{df_1min.index.min()} -> {df_1min.index.max()}\n")

    frames = []
    for tf in TIMEFRAMES:
        d = build_metrics(df_1min, tf)
        frames.append(d)
        print(f"  {tf:>4}M  {len(d):>8,} ranges | "
              f"closes outside {100*d['resolved'].mean():>4.1f}% | "
              f"touches a level {100*d['touched'].mean():>4.1f}% | "
              f"high first {100*d['hi_first'].mean():>4.1f}%")
    D = pd.concat(frames, ignore_index=True)

    grains = {
        "by_open_position": ["timeframe", "op_bucket"],
        "by_slot": ["timeframe", "slot_ET"],
        "by_slot_and_open_position": ["timeframe", "slot_ET", "op_bucket"],
    }
    tables = {}
    for name, keys in grains.items():
        t = add_stability(D, keys, aggregate(D, keys))
        t = sort_tf(t)
        tables[name] = t
        p = f"{OUT_DIR}/{SYMBOL.lower()}_stats_{name}.csv"
        t.to_csv(p, index=False, float_format="%.2f")
        print(f"\n{name}: {len(t):,} rows, {len(t.columns)} columns -> {p}")

    if WRITE_EXCEL:
        xl = f"{OUT_DIR}/{SYMBOL.lower()}_range_stats_full.xlsx"
        write_excel(tables, xl, SYMBOL)
        print(f"\nworkbook -> {xl}")

    if PRINT_SUMMARY:
        t = tables["by_open_position"]
        for metric, title in [
            ("p_hi_first_given_touch", "P(prior HIGH touched first | one was touched) %"),
            ("own_high_first_pct", "Range's own HIGH formed before its own LOW %"),
            ("hs_ret_curr_open", "After sweeping prior high: retraced to this range's open %"),
            ("ls_ret_curr_open", "After sweeping prior low: retraced to this range's open %"),
            ("hs_reversed", "After sweeping prior high: closed back below prior mid %"),
            ("ls_reversed", "After sweeping prior low: closed back above prior mid %"),
        ]:
            print("\n" + "=" * 74)
            print(title)
            print("=" * 74)
            piv = t.pivot(index="op_bucket", columns="timeframe", values=metric)
            piv = piv.reindex([l for l in OP_LABELS if l in piv.index])
            piv = piv[[f"{x}M" for x in TIMEFRAMES if f"{x}M" in piv.columns]]
            print(piv.round(1).to_string(na_rep="  -"))

    if PRINT_BY_SLOT:
        print_by_slot(tables["by_slot"])

    if PRINT_SLOT_GRIDS:
        print_slot_grids(tables["by_slot_and_open_position"])

    return tables


if __name__ == "__main__":
    main()

Loading C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv ...
  4,161,727 1-minute bars, 2013-01-02 06:00:00 -> 2025-03-14 16:59:00

    15M   273,275 ranges | closes outside 45.7% | touches a level 82.1% | high first 42.5%
    30M   132,944 ranges | closes outside 46.1% | touches a level 82.6% | high first 43.2%
    60M    67,179 ranges | closes outside 47.0% | touches a level 82.7% | high first 44.1%
   120M    30,659 ranges | closes outside 49.8% | touches a level 84.3% | high first 45.8%
   240M    17,739 ranges | closes outside 49.2% | touches a level 81.6% | high first 45.0%

by_open_position: 59 rows, 38 columns -> ./nq_stats_by_open_position.csv

by_slot: 175 rows, 38 columns -> ./nq_stats_by_slot.csv

by_slot_and_open_position: 1,854 rows, 39 columns -> ./nq_stats_by_slot_and_open_position.csv

workbook -> ./nq_range_stats_full.xlsx

P(prior HIGH touched first | one was touched) %
timeframe        15M   30M   60M  120M  240M
op_bucket                                   
below p

         ranges  up_pct  down_pct  inside_pct  p_up_given_resolved  hi_first_pct  lo_first_pct  no_touch_pct  p_hi_first_given_touch  own_high_first_pct  min_to_hi_touch  min_to_lo_touch  hs_ret_prior_mid  hs_held  hs_reversed  ls_ret_prior_mid  ls_held  ls_reversed
slot_ET                                                                                                                                                                                                                                                                   
00:00      2885    22.3      17.9        59.8                 55.5          41.7          36.5          21.8                    53.3                48.7             11.0             12.0              37.2     48.3         17.0              39.7     42.0         18.4
01:00      2891    28.1      22.9        49.0                 55.0          46.7          41.6          11.7                    52.8                48.2              9.0              9.0             

op_bucket  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                            
00:00          8.1      7.6     13.2     11.6     14.5     22.5     25.5     30.0     35.7     32.4            47.1
00:30          7.0      8.0     11.2     13.0     22.7     23.0     25.6     29.7     37.3     42.4            43.3
01:00         11.1     12.4     14.4     24.9     20.6     21.2     33.2     33.8     40.5     39.7            48.8
01:30          5.7      8.2     13.1     16.4     22.8     21.9     26.3     34.7     40.8     43.3            43.3
02:00         14.0     13.4     18.2     23.4     28.3     35.5     35.1     39.9     44.7     46.1            56.2
02:30          5.4      7.7      7.6     12.7     17.9     19.6     28.2     31.8     35.9     40.0            37.1
03:00         19.6     21.8     28.4     30.8     33.1     34.9     32.4


15M  -  closes BELOW prior low %
rows = slot (ET), columns = open position   [cells with fewer than 50 in 'ranges' are blank]
op_bucket  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                            
00:00         42.2     31.8     28.3     27.8     24.7     19.7     15.9     14.3      8.3     10.0            10.6
00:15         45.3     31.2     29.0     25.1     25.5     24.0      9.8     11.0      7.0      2.9             9.7
00:30         32.6     33.8     32.9     26.6     25.8     19.2     12.8     11.4     11.3      4.5             5.3
00:45         38.6     31.6     26.4     23.7     16.8     17.6     17.4     10.6      4.2      3.4            10.1
01:00         44.2     38.4     38.1     29.0     33.9     21.1     23.2     19.4     15.8     12.5            14.9
01:15         39.9     30.9     29.7     22.7     20.7     15

op_bucket  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                            
00:00        218.0    236.0    293.0    263.0    235.0    315.0    321.0    328.0    313.0    150.0           179.0
00:15        247.0    256.0    317.0    259.0    216.0    300.0    317.0    283.0    314.0    140.0           155.0
00:30        242.0    269.0    304.0    259.0    213.0    318.0    265.0    271.0    292.0    178.0           171.0
00:45        259.0    247.0    288.0    279.0    226.0    289.0    298.0    284.0    309.0    178.0           148.0
01:00        217.0    229.0    291.0    272.0    239.0    313.0    302.0    283.0    322.0    152.0           201.0
01:15        293.0    282.0    306.0    278.0    217.0    287.0    270.0    281.0    309.0    190.0           162.0
01:30        302.0    231.0    292.0    263.0    236.0    294.0    300.0


60M  -  closes BELOW prior low %
rows = slot (ET), columns = open position   [cells with fewer than 50 in 'ranges' are blank]
op_bucket  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                            
00:00         28.1     29.1     31.6     17.9     20.8     16.8     15.7     12.8      6.1      8.3            11.0
01:00         37.0     33.9     32.5     29.7     26.1     22.5     15.4     18.1     12.4      9.1            10.4
02:00         40.2     40.1     37.7     31.2     29.9     22.3     19.6     14.2     11.4     13.0             9.8
03:00         43.9     40.4     38.2     35.2     29.6     26.4     26.6     18.9     14.1     14.1               -
04:00         46.6     43.7     25.9     22.9     21.5     17.6     13.6      9.3      7.2      3.5               -
05:00         42.7     34.2     30.8     20.9     20.1     14

... (24 outputs left out)

op_bucket  below pLow  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0
slot_ET                                                                                                        
02:00               -     33.8     30.6     35.7     33.9     35.8     33.3     27.5     31.4     31.0        -
06:00               -     27.4     29.8     33.8     29.6     27.5     31.0     36.5     33.7        -        -
10:00               -     17.1     20.2     21.4     29.8     21.4     25.2     19.1     16.9     20.4        -
14:00               -      3.6      4.4      6.3      7.5     10.3      2.9     11.7        -        -        -
18:00             1.7      2.2      2.1      4.5      5.6        -        -        -        -        -        -
22:00               -      4.7      6.2      6.3      2.0      3.6      1.4      3.2        -        -        -

  sample size (n_lo_first):
op_bucket  below pLow  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6

op_bucket  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                            
00:00        174.0    177.0    189.0    136.0     97.0     98.0     84.0     60.0        -        -               -
00:30        194.0    208.0    170.0    136.0    112.0    102.0     63.0     59.0        -        -               -
01:00        169.0    213.0    206.0    131.0    131.0    114.0    100.0     80.0        -        -               -
01:30        217.0    207.0    187.0    136.0     94.0     87.0     75.0     55.0        -        -               -
02:00        212.0    224.0    209.0    170.0    113.0    108.0    106.0     81.0        -        -               -
02:30        212.0    219.0    194.0    141.0    105.0     93.0     59.0        -        -        -               -
03:00        247.0    262.0    253.0    194.0    128.0    135.0    113.0

op_bucket  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                            
00:00         18.3     19.9     11.4     17.6     18.9     13.0     17.4     19.4        -        -               -
00:15         13.7      9.9     14.9     16.0     11.7     13.3     25.4     14.0        -        -               -
00:30         23.0     12.8     17.3     19.3     13.7     18.0     21.3        -        -        -               -
00:45         14.5     18.6     18.2     14.2     19.3     11.3     10.7        -        -        -               -
01:00         21.3     25.6     18.5     25.2     17.2     27.6     23.9     14.9     13.6        -               -
01:15         17.8      9.7     14.0     18.4     19.0     14.0     14.5        -        -        -               -
01:30         16.8     17.7     12.9     19.3     17.9     10.7     14.7

op_bucket  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot_ET                                                                                                            
00:00        156.0    192.0    179.0    132.0    113.0    104.0     69.0     50.0        -        -               -
02:00        199.0    235.0    215.0    179.0    138.0    138.0     98.0     80.0        -        -               -
04:00        222.0    229.0    187.0    152.0    130.0    110.0     66.0     57.0        -        -               -
06:00        197.0    207.0    199.0    129.0    108.0    101.0     78.0     54.0        -        -               -
08:00        215.0    258.0    254.0    172.0    139.0    137.0     99.0     88.0     62.0        -               -
10:00        259.0    287.0    203.0    155.0    131.0     96.0     80.0     67.0        -        -               -
12:00        221.0    179.0    162.0    107.0     80.0     85.0     62.0

In [5]:
"""
Complete train/test results: every timeframe, every time of day, every open
position, both directions, nothing filtered out.
================================================================================

One row per (timeframe, slot, open-position bucket). Empty cells are still
emitted with n = 0 so nothing is silently absent.

COLUMNS
-------
  timeframe, slot, op_bucket           the cell

  train_n / test_n                     ranges in each half
  train_resolve_pct / test_resolve_pct share closing outside the prior range
  train_up_pct / test_up_pct           share closing ABOVE prior high (of all)
  train_down_pct / test_down_pct       share closing BELOW prior low  (of all)
  train_inside_pct / test_inside_pct   share closing inside (of all)

  train_res_n / test_res_n             resolved ranges in each half
  train_bull / test_bull               P(up | resolved)
  train_bear / test_bear               P(down | resolved) = 100 - bull
  train_bull_shrunk                    P(up | resolved), shrunk toward the
                                       pooled open-position curve
  train_bear_shrunk                    100 - train_bull_shrunk
  train_bull_ci_lo / _hi               95% Wilson interval on train_bull

  drift_bull                           test_bull - train_bull_shrunk
  reliable                             True when both halves clear MIN_RELIABLE

USE
---
  train_bull_shrunk is the number to build on: the raw cell rate overfits on
  thin cells, the shrunk one does not.
  Multiply by resolve_pct to get the unconditional probability - roughly half
  of all ranges close inside the prior range and resolve neither way.

Usage: edit CONFIG, then  python full_train_test.py
"""

import numpy as np
import pandas as pd

# ============================================================================
# CONFIG
# ============================================================================

CSV_PATH = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
SYMBOL = "NQ"
OUT_DIR = "."
START_DATE = "2013-01-01"
SPLIT_DATE = "2019-06-01"          # train before, test from here

TIMEFRAMES = [15, 30, 60, 120, 240]
SESSION_START_HOUR = 18
MIN_COVERAGE = 0.60

OP_EDGES = [-np.inf, 0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0, np.inf]
OP_LABELS = ["below pLow", "0.0-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4", "0.4-0.5",
             "0.5-0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9", "0.9-1.0", "pHigh or above"]

SHRINK_K = 150       # shrinkage strength toward the pooled open-position curve
MIN_RELIABLE = 50    # 'reliable' flag needs this many RESOLVED in both halves
PRINT_GRIDS = True
GRID_MIN_N = 30      # blank a printed grid cell below this many resolved


# ============================================================================
# DATA
# ============================================================================

def load_1min(path=CSV_PATH, start=START_DATE):
    df = pd.read_csv(path)
    df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"])
    df = df.set_index("Datetime").sort_index()
    return df[["Open", "High", "Low", "Close"]].loc[start:]


def build_ranges(df_1min, tf):
    first = df_1min.index[0]
    origin = pd.Timestamp(first.year, first.month, first.day, SESSION_START_HOUR)
    if origin > first:
        origin -= pd.Timedelta(days=1)
    rs = df_1min.resample(f"{tf}min", origin=origin)
    b = rs.agg({"Open": "first", "High": "max", "Low": "min", "Close": "last"})
    cov = rs["Close"].count() / float(tf)
    b = b.dropna(subset=["Open", "High", "Low", "Close"])
    b = b[cov.reindex(b.index) >= MIN_COVERAGE].copy()
    idx = pd.Series(b.index, index=b.index)
    b["prev_adjacent"] = (idx - idx.shift(1)) == pd.Timedelta(minutes=tf)
    return b


def prep(df_1min, tf):
    src = build_ranges(df_1min, tf)
    d = pd.DataFrame({"O": src["Open"], "C": src["Close"],
                      "pH": src["High"].shift(1), "pL": src["Low"].shift(1)})
    d["pR"] = d["pH"] - d["pL"]
    d = d[src["prev_adjacent"].fillna(False)].dropna()
    d = d[d["pR"] > 0]
    d["OP"] = (d["O"] - d["pL"]) / d["pR"]
    d["up"] = d["C"] > d["pH"]
    d["down"] = d["C"] < d["pL"]
    d["resolved"] = d["up"] | d["down"]
    d["op_bucket"] = pd.cut(d["OP"], OP_EDGES, labels=OP_LABELS, right=False)
    d["slot"] = d.index.strftime("%H:%M")
    return d


# ============================================================================
# AGGREGATION
# ============================================================================

def wilson(k, n, z=1.96):
    k = np.asarray(k, float)
    n = np.asarray(n, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(n > 0, k / n, np.nan)
        den = 1 + z * z / n
        c = (p + z * z / (2 * n)) / den
        h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h


def half_stats(d, keys, tag):
    """Counts and rates for one half of the sample."""
    g = d.groupby(keys, observed=False)
    out = pd.DataFrame({
        f"{tag}_n": g.size(),
        f"{tag}_up_n": g["up"].sum(),
        f"{tag}_down_n": g["down"].sum(),
        f"{tag}_res_n": g["resolved"].sum(),
    })
    n = out[f"{tag}_n"].to_numpy(float)
    r = out[f"{tag}_res_n"].to_numpy(float)
    u = out[f"{tag}_up_n"].to_numpy(float)
    dn = out[f"{tag}_down_n"].to_numpy(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        out[f"{tag}_up_pct"] = np.where(n > 0, 100 * u / n, np.nan)
        out[f"{tag}_down_pct"] = np.where(n > 0, 100 * dn / n, np.nan)
        out[f"{tag}_resolve_pct"] = np.where(n > 0, 100 * r / n, np.nan)
        bull = np.where(r > 0, 100 * u / r, np.nan)
    out[f"{tag}_inside_pct"] = 100 - out[f"{tag}_resolve_pct"]
    out[f"{tag}_bull"] = bull
    out[f"{tag}_bear"] = 100 - bull
    return out


def build_table(d, tf):
    """Full grid for one timeframe - every slot x bucket, nothing dropped."""
    slots = sorted(d["slot"].unique())
    full = pd.MultiIndex.from_product([slots, OP_LABELS], names=["slot", "op_bucket"])

    tr = d[d.index < SPLIT_DATE]
    te = d[d.index >= SPLIT_DATE]
    t = (half_stats(tr, ["slot", "op_bucket"], "train")
         .reindex(full)
         .join(half_stats(te, ["slot", "op_bucket"], "test").reindex(full)))

    # pooled open-position curve from the TRAIN half only, for shrinkage
    pool = half_stats(tr, ["op_bucket"], "train")["train_bull"]
    base = 100 * tr["up"][tr["resolved"]].mean()
    pooled = (t.index.get_level_values("op_bucket")
              .map(pool).to_numpy(float))
    pooled = np.where(np.isnan(pooled), base, pooled)
    t["train_pooled_bull"] = pooled

    r = t["train_res_n"].fillna(0).to_numpy(float)
    w = r / (r + SHRINK_K)
    raw = t["train_bull"].to_numpy(float)
    t["train_bull_shrunk"] = w * np.where(np.isnan(raw), pooled, raw) + (1 - w) * pooled
    t["train_bear_shrunk"] = 100 - t["train_bull_shrunk"]

    lo, hi = wilson(t["train_up_n"].fillna(0), t["train_res_n"].fillna(0))
    t["train_bull_ci_lo"] = 100 * lo
    t["train_bull_ci_hi"] = 100 * hi

    t["drift_bull"] = t["test_bull"] - t["train_bull_shrunk"]
    t["reliable"] = ((t["train_res_n"].fillna(0) >= MIN_RELIABLE) &
                     (t["test_res_n"].fillna(0) >= MIN_RELIABLE))

    t = t.reset_index()
    t.insert(0, "timeframe", f"{tf}M")
    for c in t.columns:
        if c.endswith("_n"):
            t[c] = t[c].fillna(0).astype(int)
    return t


COLS = ["timeframe", "slot", "op_bucket",
        "train_n", "train_res_n", "train_resolve_pct",
        "train_up_pct", "train_down_pct", "train_inside_pct",
        "train_bull", "train_bear", "train_bull_shrunk", "train_bear_shrunk",
        "train_bull_ci_lo", "train_bull_ci_hi",
        "test_n", "test_res_n", "test_resolve_pct",
        "test_up_pct", "test_down_pct", "test_inside_pct",
        "test_bull", "test_bear",
        "drift_bull", "reliable"]


def print_grids(T):
    for tf in TIMEFRAMES:
        s = T[T.timeframe == f"{tf}M"]
        if s.empty:
            continue
        for label, col, nsrc, dp in [
            ("TRAIN  P(up | resolved) %, shrunk", "train_bull_shrunk", "train_res_n", 1),
            ("TRAIN  P(down | resolved) %, shrunk", "train_bear_shrunk", "train_res_n", 1),
            ("TEST   P(up | resolved) %", "test_bull", "test_res_n", 1),
            ("TEST   P(down | resolved) %", "test_bear", "test_res_n", 1),
            ("TEST minus TRAIN, points", "drift_bull", "test_res_n", 1),
            ("resolve rate %, train", "train_resolve_pct", "train_n", 1),
            ("resolved sample size, train", "train_res_n", "train_res_n", 0),
            ("resolved sample size, test", "test_res_n", "test_res_n", 0),
        ]:
            piv = s.pivot(index="slot", columns="op_bucket", values=col)
            n = s.pivot(index="slot", columns="op_bucket", values=nsrc)
            cols = [c for c in OP_LABELS if c in piv.columns]
            piv, n = piv[cols].sort_index(), n[cols].sort_index()
            piv = piv.where(n >= GRID_MIN_N)
            if piv.notna().sum().sum() == 0:
                continue
            print("\n" + "=" * 110)
            print(f"{tf}M  -  {label}    rows = slot (ET), columns = open position")
            print(f"cells with fewer than {GRID_MIN_N} in '{nsrc}' are blank")
            print("=" * 110)
            print(piv.round(dp).to_string(na_rep="    -"))


def main():
    print(f"Loading {CSV_PATH} ...")
    df_1min = load_1min()
    print(f"  {len(df_1min):,} 1-minute bars, "
          f"{df_1min.index.min()} -> {df_1min.index.max()}")
    print(f"  train < {SPLIT_DATE} <= test\n")

    tables = []
    for tf in TIMEFRAMES:
        d = prep(df_1min, tf)
        t = build_table(d, tf)
        tables.append(t)
        rel = int(t["reliable"].sum())
        print(f"  {tf:>4}M  {len(d):>8,} ranges | {t['slot'].nunique():>3} slots | "
              f"{len(t):>5} cells | {rel:>4} reliable "
              f"(>={MIN_RELIABLE} resolved both halves)")

    T = pd.concat(tables, ignore_index=True)[COLS]
    path = f"{OUT_DIR}/{SYMBOL.lower()}_full_train_test.csv"
    T.to_csv(path, index=False, float_format="%.2f")
    print(f"\n{len(T):,} rows -> {path}")

    rel = T[T.reliable]
    print(f"\n{len(rel):,} reliable cells. On those, correlation between the train "
          f"estimate and\nthe test outcome is "
          f"{rel.train_bull_shrunk.corr(rel.test_bull):.3f}, mean absolute drift "
          f"{rel.drift_bull.abs().mean():.1f} points.")

    print("\nPOOLED BY OPEN POSITION (all slots, all reliable and unreliable cells)")
    print(f"{'bucket':<16}" + "".join(f"{str(t)+'M':>16}" for t in TIMEFRAMES))
    print(f"{'':<16}" + "".join(f"{'train / test':>16}" for _ in TIMEFRAMES))
    for b in OP_LABELS:
        row = f"{b:<16}"
        for tf in TIMEFRAMES:
            s = T[(T.timeframe == f"{tf}M") & (T.op_bucket == b)]
            trn, tst = s.train_res_n.sum(), s.test_res_n.sum()
            if trn < 30 or tst < 30:
                row += f"{'-':>16}"
                continue
            bt = 100 * (s.train_up_pct * s.train_n / 100).sum() / trn
            bs = 100 * (s.test_up_pct * s.test_n / 100).sum() / tst
            row += f"{f'{bt:.1f} / {bs:.1f}':>16}"
        print(row)

    if PRINT_GRIDS:
        print_grids(T)

    return T


if __name__ == "__main__":
    main()

Loading C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv ...
  4,161,727 1-minute bars, 2013-01-02 06:00:00 -> 2025-03-14 16:59:00
  train < 2019-06-01 <= test

    15M   273,275 ranges |  92 slots |  1104 cells |  678 reliable (>=50 resolved both halves)
    30M   132,944 ranges |  45 slots |   540 cells |  348 reliable (>=50 resolved both halves)
    60M    67,179 ranges |  22 slots |   264 cells |  168 reliable (>=50 resolved both halves)
   120M    30,659 ranges |  10 slots |   120 cells |   76 reliable (>=50 resolved both halves)
   240M    17,739 ranges |   6 slots |    72 cells |   37 reliable (>=50 resolved both halves)

2,100 rows -> ./nq_full_train_test.csv

1,307 reliable cells. On those, correlation between the train estimate and
the test outcome is 0.967, mean absolute drift 5.2 points.

POOLED BY OPEN POSITION (all slots, all reliable and unreliable cells)
bucket                       15M             30M             60M            120M            240M
                   

op_bucket  below pLow  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot                                                                                                                           
00:00               -     93.2     76.7     75.4     57.4     54.1     46.4     34.2     32.6     15.8     26.0               -
00:15               -     94.4     82.6     76.8     74.6     58.2     53.4     29.3     29.3     15.8      8.0               -
00:30               -     76.8     70.6     69.7     58.3     53.8     46.4     35.2     16.5     18.8     11.7               -
00:45               -     89.4     88.2     68.3     62.1     50.9     55.2     42.9     25.0      7.0      8.3               -
01:00               -     74.5     65.4     72.6     63.8     56.7     40.8     45.3     43.9     29.6     25.5               -
01:15               -     81.0     91.1     73.9     69.2     54.9     38.3     36.8     20.3     20.2  

op_bucket  below pLow  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot                                                                                                                           
00:00               -     44.0     60.0     65.0     61.0     74.0     69.0     73.0     89.0     76.0     50.0               -
00:15               -     54.0     46.0     69.0     63.0     55.0     73.0     58.0     75.0     76.0     50.0               -
00:30               -     56.0     68.0     89.0     60.0     65.0     84.0     54.0     79.0     80.0     60.0               -
00:45               -     47.0     51.0     60.0     66.0     53.0     58.0     63.0     72.0     86.0     60.0               -
01:00               -     55.0     78.0     95.0     80.0     90.0     76.0     75.0     98.0     98.0     51.0               -
01:15               -     58.0     56.0     69.0     52.0     51.0     47.0     57.0     69.0     99.0  


30M  -  resolve rate %, train    rows = slot (ET), columns = open position
cells with fewer than 30 in 'train_n' are blank
op_bucket  below pLow  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot                                                                                                                           
00:00               -     40.9     28.6     43.9     29.8     32.2     39.0     40.7     36.0     44.0     35.2            56.8
00:30               -     38.3     40.5     36.4     33.3     48.2     36.5     36.8     38.2     42.4     41.5            54.0
01:00               -     54.7     51.6     45.9     52.8     42.3     45.6     49.6     47.7     51.1     41.6            58.4
01:30               -     44.9     40.6     39.7     39.6     33.9     31.5     42.4     42.3     45.2     43.1            53.2
02:00               -     56.5     52.7     57.3     54.7     54.9     58.2     57.4     63.8     54.4     5


60M  -  resolved sample size, train    rows = slot (ET), columns = open position
cells with fewer than 30 in 'train_res_n' are blank
op_bucket  below pLow  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot                                                                                                                           
00:00               -        -       38       48     39.0     41.0     61.0       60       53       73       35            35.0
01:00               -     39.0       50       75     60.0     56.0     84.0       69       63       92       49            32.0
02:00               -     71.0       73       71     68.0     62.0     78.0       79       83      108       55            44.0
03:00               -     92.0       89       92     92.0     81.0    106.0       93       91      112       82            36.0
04:00               -     80.0       80       52     66.0     55.0     64.0       61       60     


240M  -  TEST minus TRAIN, points    rows = slot (ET), columns = open position
cells with fewer than 30 in 'test_res_n' are blank
op_bucket  below pLow  0.0-0.1  0.1-0.2  0.2-0.3  0.3-0.4  0.4-0.5  0.5-0.6  0.6-0.7  0.7-0.8  0.8-0.9  0.9-1.0  pHigh or above
slot                                                                                                                           
02:00               -     15.8      1.5     -1.9     12.9     -8.7      2.9      2.5      7.7    -10.9     -8.8               -
06:00               -      4.7      0.2     -4.2      5.4      5.6      0.7      3.7     -3.3    -13.0     -8.0               -
10:00               -      0.1    -11.3     -6.5     -1.6      1.7     -4.7     -5.6     -7.7      0.4      3.1               -
14:00               -    -15.0    -15.8     -9.0     -6.9      9.6     -6.4     11.3      0.6      7.5      3.5               -
18:00               -        -    -22.1        -        -        -        -        -     13.7      4.